# Inference from any pose

With this notebook you can run inference of HOOD starting with a garment in any pose. For the body sequence you can use either:
* Sequence of SMPL parameters (see `$HOOD_DATA/fromanypose/pose_sequence.pkl` as an example) or
* Sequence of meshes (see `$HOOD_DATA/fromanypose/mesh_sequence.pkl` as an example)

You also need to have a garment mesh (as `.obj` file) which is aligned with the first frame of your body sequence.

In this notebook we first show how to convert a garment mesh in `.obj` format into a `.pkl` template used in HOOD. 

Then, we show how to use the configuration file `aux/from_any_pose.yaml` to run inference with over albitrary SMPL sequence and arbitrary mesh sequence.

Note that, pinned vertices are not yet supported in this example, so do not expect it to work with lower body garments (they would slide down :c)

基于任意姿势的推理

通过此笔记本，您可以针对任意姿势的服装运行 HOOD 推理。对于身体序列，您可以使用以下任一方式：
SMPL 参数序列（参见 $HOOD_DATA/fromanypose/pose_sequence.pkl作为示例）或
网格序列（参见$HOOD_DATA/fromanypose/mesh_sequence.pkl作为示例）
您还需要准备一个服装网格（作为.obj文件），该网格需与身体序列的第一帧对齐。
在本笔记本中，我们将首先演示如何将.obj格式的服装网格转换为HOOD中使用的.pkl模板。
然后，我们将演示如何使用配置文件 aux/from_any_pose.yaml对任意 SMPL 序列和任意网格序列进行推理。
请注意，本示例目前尚不支持固定顶点，因此不要指望它能对下半身服装生效（它们会滑落下来 :c）

## Set environmental variables

In [1]:
import os

HOOD_PROJECT = "/home/zhudf/projects/HOOD/"
HOOD_DATA = "/home/zhudf/data/hood_data/hood_data/"

os.environ["HOOD_PROJECT"] = HOOD_PROJECT
os.environ["HOOD_DATA"] = HOOD_DATA

## Create template file from the .obj file:

从 .obj 文件创建模板文件：

使用 utils.mesh_creation::obj2template() 函数将 .obj 文件转换为模板字典，然后使用 pickle_dump 将其保存
$HOOD_DATA/fromanypose/tshirt.obj 作为示例提供

Use `utils.mesh_creation::obj2template()` function to convert an `.obj` file into a template dictionary and then save it with `pickle_dump`

`$HOOD_DATA/fromanypose/tshirt.obj` is provided as an example

In [2]:
import trimesh
from utils.mesh_creation import obj2template
from pathlib import Path
from utils.common import pickle_dump
from utils.defaults import DEFAULTS

In [ ]:
obj_path = Path(DEFAULTS.data_root) / 'fromanypose' / 'tshirt.obj'
out_template_path = Path(DEFAULTS.data_root) / 'fromanypose' / 'tshirt.pkl'

template_dict = obj2template(obj_path)
pickle_dump(template_dict, out_template_path)

## Inference with a SMPL sequence

Here we use a sequence of SMPL parameters as a body sequence. The garment template needs to be aligned with the first frame of the sequence.

The SMPL pose sequence has to be a `.pkl` file containing a dictionary of the following items (see `$HOOD_DATA/fromanypose/pose_sequence.pkl` as an example):
* `body_pose`: np.array [N, 69]
* `global_orient`: np.array [N, 3]
* `transl`: np.array [N, 3]
* `betas`: np.array [10,]

### Edit `aux/from_any_pose` config

First, change the configuration file `configs/aux/from_any_pose.yaml` to set your garment template and the pose sequence

Go to `configs/aux/from_any_pose.yaml` and edit following fields there:

- `dataloader.from_any_pose.scanreg.pose_sequence_type`, type of the pose sequence can be either `smpl` or `mesh`. For this example, set `smpl`
- `dataloader.from_any_pose.scanreg.pose_sequence_path` path to the smpl pose sequence relative to `$HOOD_DATA`. For this example we use `fromanypose/pose_sequence.pkl`
- `dataloader.from_any_pose.scanreg.garment_template_path` path to the .pkl file with the garment template  For this example we use `fromanypose/tshirt.pkl`

使用 SMPL 序列进行推理
在此，我们将一组 SMPL 参数序列用作身体序列。服装模板需要与该序列的第一个帧对齐。
SMPL 姿势序列必须是一个 .pkl 文件，其中包含一个包含以下项的字典（参见 $HOOD_DATA/fromanypose/pose_sequence.pkl 作为示例）：
• body_pose: np.array [N, 69]
• global_orient: np.array [N, 3]
• transl: np.array [N, 3]
• betas: np.array [10,]

编辑 aux/from_any_pose 配置

首先，修改配置文件 configs/aux/from_any_pose.yaml 以设置您的服装模板和姿势序列

打开 configs/aux/from_any_pose.yaml 并编辑其中的以下字段：
• dataloader.from_any_pose.scanreg.pose_sequence_type，姿势序列类型可以是 smpl 或 mesh。在本示例中，请设置为 smpl
• dataloader.from_any_pose.scanreg.pose_sequence_path，相对于 $HOOD_DATA 的 smpl 姿势序列路径。在本示例中，我们使用 fromanypose/pose_sequence.pkl
• dataloader.from_any_pose.scanreg.garment_template_path：服装模板 .pkl 文件的路径。本例中使用 fromanypose/tshirt.pkl

Inference.ipynb 之所以不会要求你手工把服装模板对齐到序列第一帧，是因为它走的是另一套流程：
先从 garments_dict.pkl 读取规范姿态下的服装模板和 LBS 数据，然后 Dataset 在构造 sample 时，利用 SMPL 的第一帧参数 + LBS，把服装自动变换到第一帧人体附近。

In [7]:
# ============================================================
# 1. 导入需要的库
# ============================================================

import pickle
import torch
import smplx

from utils.datasets import load_garments_dict
from utils.garment_smpl import GarmentSMPL

In [8]:
# ============================================================
# 2. 设置路径
# ============================================================

smpl_path = "/home/zhudf/data/hood_data/hood_data/aux_data/smpl/SMPL_NEUTRAL.pkl"

garments_dict_path = "/home/zhudf/data/hood_data/hood_data/aux_data/garments_dict.pkl"

pose_path = "/home/zhudf/data/hood_data/hood_data/fromanypose/pose_sequence.pkl"

In [9]:
# ============================================================
# 3. 加载 SMPL 模型
# ============================================================

smpl_model = smplx.SMPL(
    model_path=smpl_path,
    gender="neutral"
)

print("SMPL loaded")

SMPL loaded


In [10]:
# ============================================================
# 4. 加载 garments_dict.pkl
#
# load_garments_dict() 会自动把：
# shapedirs
# posedirs
# lbs_weights
# v
#
# 从 numpy 转成 torch.Tensor
# ============================================================

garments_dict = load_garments_dict(garments_dict_path)

print("garments:")
print(garments_dict.keys())

garments:
dict_keys(['shortsleeve', 'pants_shorter', 'tshirt_unzipped', 'dress', 'tshirt', 'longsleeve', 'tank', 'pants', 'shorts', 'tshirt_unzipped_dynamic', 'novel_top', 'novel_tank', 'hooded_tight_dress', 'tight_dress', 'dress_split', 'longskirt', 'tank_and_shorts', 'tank_and_skirt', 'cindy_020_combined_test', 'aaron_022::top_coat', 'aaron_022::bottom_pants', 'aaron_022::top_blouse', 'ben_004::scarf', 'ben_004::top_coat', 'ben_004::bottom_pants', 'ben_004::top_vest', 'ben_004::top_tshirt', 'celina_002::bottom_pants', 'celina_002::top_jacket', 'celina_002::top_blouse', 'caren_008::top_overcoat', 'caren_008::dress', 'cindy_020::bottom_skirt', 'cindy_020::top_blouse', 'aaron_013::bottom_pants', 'aaron_013::top_jacket', 'aaron_022::top_coat_buttoned', 'aaron_022_combined', 'ben_004_combined', 'caren_008_combined', 'celina_002_combined', 'cindy_020_combined'])


In [12]:
# ============================================================
# 5. 取得这件服装的 garment_dict
# ============================================================

garment_dict = garments_dict['tshirt']

print(garment_dict.keys())

print("\nLBS keys:")
print(garment_dict["lbs"].keys())

dict_keys(['rest_pos', 'faces', 'node_type', 'lbs', 'center', 'coarse_edges', 'gender', 'model_type'])

LBS keys:
dict_keys(['v', 'f', 'shapedirs', 'posedirs', 'lbs_weights'])


In [13]:
# ============================================================
# 6. 加载 SMPL pose sequence
# ============================================================

with open(pose_path, "rb") as f:
    pose_sequence = pickle.load(f)

print(pose_sequence.keys())

print("body_pose:", pose_sequence["body_pose"].shape)
print("global_orient:", pose_sequence["global_orient"].shape)
print("transl:", pose_sequence["transl"].shape)
print("betas:", pose_sequence["betas"].shape)

dict_keys(['body_pose', 'global_orient', 'transl', 'betas'])
body_pose: (587, 69)
global_orient: (587, 3)
transl: (587, 3)
betas: (10,)


In [14]:
# ============================================================
# 7. 取第 0 帧
# ============================================================

body_pose0 = torch.tensor(
    pose_sequence["body_pose"][0],
    dtype=torch.float32
).unsqueeze(0)

global_orient0 = torch.tensor(
    pose_sequence["global_orient"][0],
    dtype=torch.float32
).unsqueeze(0)

transl0 = torch.tensor(
    pose_sequence["transl"][0],
    dtype=torch.float32
).unsqueeze(0)

betas = torch.tensor(
    pose_sequence["betas"],
    dtype=torch.float32
).reshape(1, -1)

print("body_pose0:", body_pose0.shape)
print("global_orient0:", global_orient0.shape)
print("transl0:", transl0.shape)
print("betas:", betas.shape)

body_pose0: torch.Size([1, 69])
global_orient0: torch.Size([1, 3])
transl0: torch.Size([1, 3])
betas: torch.Size([1, 10])


In [15]:
# ============================================================
# 8. SMPL 完整 pose
#
# global_orient 3维
# +
# body_pose 69维
# =
# full_pose 72维
# ============================================================

full_pose0 = torch.cat(
    [global_orient0, body_pose0],
    dim=1
)

print("full_pose0:", full_pose0.shape)

full_pose0: torch.Size([1, 72])


In [16]:
# ============================================================
# 9. 创建 GarmentSMPL
#
# 这个类就是 HOOD 专门负责：
# canonical garment
#       ↓
# SMPL shape + pose
#       ↓
# LBS
#       ↓
# posed garment
# ============================================================

garment_smpl = GarmentSMPL(
    smpl_model,
    garment_dict["lbs"]
)

In [17]:
# ============================================================
# 10. 把 canonical garment LBS 到 pose sequence 第0帧
# ============================================================

with torch.no_grad():

    garment_frame0 = garment_smpl.make_vertices(
        betas=betas,
        full_pose=full_pose0,
        transl=transl0
    )

print("garment_frame0 shape:", garment_frame0.shape)

garment_frame0 shape: torch.Size([1, 4424, 3])


In [18]:
# ============================================================
# 13. 构造 from_any_pose garment template
# ============================================================

import numpy as np

template = {
    "vertices": garment_frame0[0].cpu().numpy(),

    # 使用 garments_dict 中原始服装的 faces
    "faces": np.asarray(garment_dict["faces"])
}

In [1]:
# ============================================================
# 14. 添加 hierarchical coarse edges
# ============================================================
from utils.mesh_creation import add_coarse_edges
template = add_coarse_edges(
    template,
    4
)

KeyError: 'HOOD_PROJECT'

### Load runner and dataloader from `from_any_pose` config

In [4]:
from utils.validation import Config as ValidationConfig
from utils.arguments import load_params, create_modules
from utils.arguments import load_params
from utils.common import move2device, pickle_dump
from utils.defaults import DEFAULTS
from pathlib import Path
import torch

In [5]:
modules, config = load_params('aux/from_any_pose')
dataloader_m, runner_module, runner, aux_modules = create_modules(modules, config)
dataloader = dataloader_m.create_dataloader()

checkpoint_path = Path(DEFAULTS.data_root) / 'trained_models' / 'postcvpr.pth'
state_dict =  torch.load(checkpoint_path)
runner.load_state_dict(state_dict['training_module'])

<All keys matched successfully>

### load sample, infer and save trajectories

To visualise saved trajectories, see `Inference.ipynb::write_video`

In [6]:
sample = next(iter(dataloader))

trajectories_dict = runner.valid_rollout(sample)
# Save the sequence to disc
out_path = Path(DEFAULTS.data_root) / 'temp' / 'output_n.pkl'
print(f"Rollout saved into {out_path}")
pickle_dump(dict(trajectories_dict), out_path)

100%|██████████████████████████████████████████████████████████████| 386/386 [00:25<00:00, 15.08it/s]

Rollout saved into /home/zhudf/data/hood_data/hood_data/temp/output_n.pkl


## Inference with a mesh sequence

Here we use a sequence of arbitrary meshes.  The garment template needs to be aligned with the first frame of the sequence.

The mesh sequence has to be a `.pkl` file containing a dictionary of the following items (see `$HOOD_DATA/fromanypose/mesh_sequence.pkl` as an example):
* `verts`: np.array [N, 3]
* `faces`: np.array [F, 3]

### Edit `aux/from_any_pose` config

Change the configuration file `configs/aux/from_any_pose.yaml` to set your garment template and the pose sequence

Go to `configs/aux/from_any_pose.yaml` and edit following fields there:

- `dataloader.from_any_pose.scanreg.pose_sequence_type`, type of the pose sequence can be either `smpl` or `mesh`. For this example, set `mesh`
- `dataloader.from_any_pose.scanreg.pose_sequence_path` path to the smpl pose sequence relative to `$HOOD_DATA`. For this example we use `fromanypose/mesh_sequence.pkl`
- `dataloader.from_any_pose.scanreg.garment_template_path` path to the .pkl file with the garment template  For this example we use `fromanypose/tshirt.pkl`

### Load runner and dataloader from `from_any_pose` config

In [ ]:
from utils.validation import Config as ValidationConfig
from utils.arguments import load_params, create_modules
from utils.arguments import load_params
from utils.common import move2device, pickle_dump
from utils.defaults import DEFAULTS
from pathlib import Path
import torch

In [ ]:
modules, config = load_params('aux/from_any_pose')
dataloader_m, runner_module, runner, aux_modules = create_modules(modules, config)
dataloader = dataloader_m.create_dataloader()

checkpoint_path = '/mnt/sdb1/hood_public/trained_models/postcvpr.pth'
state_dict =  torch.load(checkpoint_path)
runner.load_state_dict(state_dict['training_module'])

### load sample, infer and save trajectories

To visualise saved trajectories, see `Inference.ipynb::write_video`

In [ ]:
sample = next(iter(dataloader))

trajectories_dict = runner.valid_rollout(sample)
# Save the sequence to disc
out_path = Path(DEFAULTS.data_root) / 'temp' / 'output_m.pkl'
print(f"Rollout saved into {out_path}")
pickle_dump(dict(trajectories_dict), out_path)